In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset   
from torch.utils.data import DataLoader     
from sklearn.datasets import make_classification  
from sklearn.model_selection import train_test_split


In [2]:
torch.manual_seed(42)
np.random.seed(42)


In [3]:
X, y = make_classification(
    n_samples=500,                          # মোট 500টি sample
    n_features=2,                           # প্রতিটি sample-এ 2টি feature
    n_informative=2,                        # দুটি feature-ই class নির্ধারণে তথ্যবহুল
    n_redundant=0,                          # অতিরিক্ত redundant feature নেই
    n_classes=2,                            # দুটি class: 0 এবং 1
    n_clusters_per_class=1,                 # প্রতি class-এ একটি cluster
    class_sep=1.5,                          # Class দুটির মধ্যে separation
    random_state=42                         # Dataset পুনরায় তৈরি করা সহজ করে
)

In [4]:
X_train, X_test, y_train, y_test = train_test_split( X,y,test_size=0.2,random_state=42,stratify=y)


In [5]:
X_train_tensor = torch.tensor( X_train, dtype=torch.float32) # Input float32 Tensor

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)  # Test input Tensor

y_train_tensor = torch.tensor( y_train, dtype=torch.float32).view(-1, 1) # Label shape: (samples, 1)

y_test_tensor = torch.tensor( y_test, dtype=torch.float32).view(-1, 1)                              


In [6]:
class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features = features            # Input features সংরক্ষণ
        self.labels = labels                # Target labels সংরক্ষণ

    def __len__(self):
        return len(self.features)           # মোট sample সংখ্যা ফেরত

    def __getitem__(self, index):
        return self.features[index], self.labels[index]
                                             # নির্দিষ্ট index-এর sample ফেরত


In [7]:
# Step 7: Training ও testing Dataset তৈরি করা
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)  # Training Dataset

test_dataset = CustomDataset(X_test_tensor, y_test_tensor) # Testing Dataset

# Step 8: DataLoader তৈরি করা
train_loader = DataLoader(
    train_dataset,
    batch_size=32,                          # প্রতি batch-এ 32টি sample
    shuffle=True                             # Training sample-এর ক্রম এলোমেলো
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,                          # প্রতি batch-এ সর্বোচ্চ 32টি sample
    shuffle=False                            # Test-এর ক্রম অপরিবর্তিত
)

In [8]:
class NN_model(nn.Module):

    def __init__(self, num_features):
        super().__init__()                  # nn.Module initialize করা

        self.network = nn.Sequential(
            nn.Linear(num_features, 3),     # 2 input -> 3 hidden neurons
            nn.ReLU(),                      # Hidden layer activation
            nn.Linear(3, 1),                # 3 hidden -> 1 output
            nn.Sigmoid()                    # Output-কে probability-তে রূপান্তর
        )

    def forward(self, features):
        return self.network(features)       # Forward propagation

In [9]:
model = NN_model( num_features=X_train_tensor.shape[1]) # Input feature সংখ্যা = 2



In [10]:
loss_function = nn.BCELoss()                # Binary Cross-Entropy loss

optimizer = torch.optim.SGD(
    model.parameters(),                     # Trainable weight ও bias
    lr=0.01                                 # Learning rate
)

In [11]:
epochs = 100

In [12]:
for epoch in range(epochs):

    model.train()                           # Training mode চালু
    total_loss = 0.0                        # মোট loss-এর প্রাথমিক মান

    for batch_features, batch_labels in train_loader:

        y_pred = model(batch_features)      # Forward propagation
        loss = loss_function(
            y_pred, batch_labels
        )                                   # Prediction-এর ভুল হিসাব

        optimizer.zero_grad()               # আগের gradient শূন্য করা
        loss.backward()                     # Backpropagation
        optimizer.step()                    # Weight ও bias আপডেট

        total_loss += loss.item() * len(batch_features)
                                             # Batch-এর মোট loss যোগ

    average_loss = total_loss / len(train_dataset)
                                             # Epoch-এর average loss

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch: {epoch + 1:3d} | "
            f"Loss: {average_loss:.4f}"
        )                                   # প্রতি 10 epoch-এ loss দেখানো

Epoch:  10 | Loss: 0.5713
Epoch:  20 | Loss: 0.4835
Epoch:  30 | Loss: 0.4205
Epoch:  40 | Loss: 0.3726
Epoch:  50 | Loss: 0.3351
Epoch:  60 | Loss: 0.3059
Epoch:  70 | Loss: 0.2827
Epoch:  80 | Loss: 0.2642
Epoch:  90 | Loss: 0.2491
Epoch: 100 | Loss: 0.2368


In [13]:
model.eval()                                # Evaluation mode চালু

correct = 0                                 # সঠিক prediction-এর সংখ্যা
total = 0                                   # মোট test sample
test_loss_total = 0.0                       # Test loss-এর মোট যোগফল

with torch.no_grad():                       # Gradient গণনা বন্ধ

    for batch_features, batch_labels in test_loader:

        probabilities = model(batch_features)
                                             # প্রতিটি sample-এর probability

        loss = loss_function(
            probabilities, batch_labels
        )                                   # Test batch-এর loss

        test_loss_total += loss.item() * len(batch_features)
                                             # Test loss যোগ

        predictions = (
            probabilities >= 0.5
        ).float()                            # Probability থেকে 0/1 class

        correct += (
            predictions == batch_labels
        ).sum().item()                      # সঠিক prediction গণনা

        total += len(batch_labels)          # Sample সংখ্যা যোগ

In [14]:
test_accuracy = 100 * correct / total      # Accuracy percentage
average_test_loss = test_loss_total / total # Average test loss

print("\n===== FINAL RESULTS =====")
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Test Loss: {average_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

# Step 15: একটি নতুন sample-এর prediction
new_sample = torch.tensor(
    [[0.5, -0.3]], dtype=torch.float32
)                                           # দুটি feature-সহ নতুন sample

model.eval()                                # Evaluation mode নিশ্চিত করা

with torch.no_grad():
    probability = model(new_sample)         # Class 1-এর probability

    predicted_class = (
        probability >= 0.5
    ).int().item()                          # Final class: 0 অথবা 1

print("\n===== NEW SAMPLE PREDICTION =====")
print(f"Class 1 Probability: {probability.item():.4f}")
print(f"Predicted Class: {predicted_class}")


===== FINAL RESULTS =====
Train samples: 400
Test samples: 100
Test Loss: 0.2135
Test Accuracy: 96.00%

===== NEW SAMPLE PREDICTION =====
Class 1 Probability: 0.2512
Predicted Class: 0
